# Notebook 2: Evaluation on a New Dataset

This notebook:
- Loads the fine-tuned Whisper model saved in Notebook 1
- Runs inference on `test-clean` (or any custom folder you specify)
- Computes **Word Error Rate (WER)** and **Character Error Rate (CER)**
- Displays a comparison table of predicted vs. ground-truth transcripts

## Step 1: Install Dependencies

In [1]:
%pip install transformers torch torchaudio evaluate jiwer soundfile librosa pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Imports and Paths

In [2]:
import torch
import torchaudio
import numpy as np
import pandas as pd
from pathlib import Path
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import Dataset, Audio
import evaluate

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR    = Path("../../../")                                       # NLP_project root
WEIGHTS_DIR = Path("../weights/whisper-base-librispeech")             # saved model
EXTRACT_DIR = BASE_DIR / "librispeech"                                # extracted audio

# Change this to any folder that contains .flac/.wav files + *.trans.txt
EVAL_SPLIT  = "test-clean"

SAMPLE_RATE = 16_000
MAX_SAMPLES = 200   # set None to evaluate all

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Step 3: Load Fine-Tuned Model

In [3]:
print(f"Loading model from: {WEIGHTS_DIR.resolve()}")
processor = WhisperProcessor.from_pretrained(str(WEIGHTS_DIR))
model     = WhisperForConditionalGeneration.from_pretrained(str(WEIGHTS_DIR)).to(device)
model.eval()
print("Model loaded successfully.")

Loading model from: E:\UMD\Data_641_PCS1\NLP_project\PronounceAI\model\module_1\weights\whisper-base-librispeech


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Model loaded successfully.


## Step 4: Load Evaluation Dataset

In [4]:
def parse_librispeech_split(root: Path, split_name: str) -> Dataset:
    """Collect (audio_path, transcript) pairs from a LibriSpeech split."""
    audio_paths, transcripts = [], []
    split_dir = root / "LibriSpeech" / split_name
    if not split_dir.exists():
        raise FileNotFoundError(f"Split not found: {split_dir}")
    for trans_file in sorted(split_dir.rglob("*.trans.txt")):
        chapter_dir = trans_file.parent
        with open(trans_file) as f:
            for line in f:
                parts = line.strip().split(" ", 1)
                if len(parts) != 2:
                    continue
                utt_id, text = parts
                flac_path = chapter_dir / f"{utt_id}.flac"
                if flac_path.exists():
                    audio_paths.append(str(flac_path))
                    transcripts.append(text.lower())
    ds = Dataset.from_dict({"audio": audio_paths, "sentence": transcripts})
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    return ds

eval_dataset = parse_librispeech_split(EXTRACT_DIR, EVAL_SPLIT)
if MAX_SAMPLES:
    eval_dataset = eval_dataset.select(range(min(MAX_SAMPLES, len(eval_dataset))))

print(f"Evaluation samples: {len(eval_dataset)}")

Evaluation samples: 200


## Step 5: Run Inference

In [5]:
def transcribe_batch(audio_arrays: list) -> list:
    """Run Whisper inference on a list of 16 kHz audio arrays."""
    inputs = processor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(inputs)

    return processor.batch_decode(predicted_ids, skip_special_tokens=True)


def load_audio(path: str) -> np.ndarray:
    """Load a .flac/.wav file with soundfile and resample to SAMPLE_RATE if needed."""
    import soundfile as sf
    import torchaudio.functional as F_audio
    array, sr = sf.read(path, dtype="float32")
    if array.ndim > 1:
        array = array.mean(axis=1)  # stereo -> mono
    if sr != SAMPLE_RATE:
        tensor = torch.from_numpy(array).unsqueeze(0)
        tensor = F_audio.resample(tensor, sr, SAMPLE_RATE)
        array = tensor.squeeze(0).numpy()
    return array


predictions, references = [], []
BATCH_SIZE = 8

raw_dataset = eval_dataset.cast_column("audio", Audio(decode=False))

for i in range(0, len(raw_dataset), BATCH_SIZE):
    batch = raw_dataset[i : i + BATCH_SIZE]
    audio_arrays = [load_audio(a["path"]) for a in batch["audio"]]
    preds = transcribe_batch(audio_arrays)
    predictions.extend(preds)
    references.extend(batch["sentence"])
    if (i // BATCH_SIZE) % 5 == 0:
        print(f"  Processed {min(i + BATCH_SIZE, len(raw_dataset))} / {len(raw_dataset)}")

print("Inference complete.")

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para

  Processed 8 / 200
  Processed 48 / 200
  Processed 88 / 200
  Processed 128 / 200
  Processed 168 / 200
Inference complete.


In [6]:
# Debug: inspect what keys the audio dict has when decode=False
raw_dataset = eval_dataset.cast_column("audio", Audio(decode=False))
sample = raw_dataset[0]
print(type(sample["audio"]))
print(sample["audio"])

<class 'dict'>
{'bytes': None, 'path': '..\\..\\..\\librispeech\\LibriSpeech\\test-clean\\1089\\134686\\1089-134686-0000.flac'}


## Step 6: Compute WER and CER

In [7]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

wer = wer_metric.compute(predictions=predictions, references=references)
cer = cer_metric.compute(predictions=predictions, references=references)

print(f"\n{'='*40}")
print(f"  Word Error Rate  (WER) : {wer:.4f}  ({wer*100:.2f}%)")
print(f"  Char Error Rate  (CER) : {cer:.4f}  ({cer*100:.2f}%)")
print(f"{'='*40}")


  Word Error Rate  (WER) : 0.0470  (4.70%)
  Char Error Rate  (CER) : 0.0160  (1.60%)


## Step 7: Show Example Predictions vs. Ground Truth

In [8]:
N_EXAMPLES = 10

results_df = pd.DataFrame({
    "Ground Truth" : references[:N_EXAMPLES],
    "Prediction"   : predictions[:N_EXAMPLES],
})

results_df["Sample WER"] = results_df.apply(
    lambda row: round(
        wer_metric.compute(predictions=[row["Prediction"]], references=[row["Ground Truth"]]), 4
    ),
    axis=1,
)

results_df["Sample CER"] = results_df.apply(
    lambda row: round(
        cer_metric.compute(predictions=[row["Prediction"]], references=[row["Ground Truth"]]), 4
    ),
    axis=1,
)

pd.set_option("display.max_colwidth", 80)
display(results_df)

,Ground Truth,Prediction,Sample WER,Sample CER
0,he hoped there would be stew for dinner turnips and carrots and bruised pota...,he hoped there would be stew for dinner turnips and carrots and bruised pota...,0.1071,0.0316
1,stuff it into you his belly counselled him,Stuffed into you his belly countled him,0.3750,0.1667
2,after early nightfall the yellow lamps would light up here and there the squ...,after early nightfall the yellow lamps would light up here and there the squ...,0.0556,0.0192
3,hello bertie any good in your mind,hello barty any good in your mind,0.1429,0.0882
4,number ten fresh nelly is waiting on you good night husband,number ten fresh nelly is waiting on you good night husband,0.0000,0.0000
5,the music came nearer and he recalled the words the words of shelley's fragm...,the music came nearer and he recalled the words the words of Shelley's fragm...,0.0455,0.0074
6,the dull light fell more faintly upon the page whereon another equation bega...,the dull light fell more faintly upon the page where on another equation beg...,0.1250,0.0214
7,a cold lucid indifference reigned in his soul,a cold lucid indifference reigned in his soul,0.0000,0.0000
8,the chaos in which his ardour extinguished itself was a cold indifferent kno...,the chaos in which his order extinguished itself was a cold in different kno...,0.2000,0.0430
9,at most by an alms given to a beggar whose blessing he fled from he might ho...,at most by an arm given to a beggar whose blessing he fled from he might hop...,0.0370,0.0149


## Step 8 (Optional): Evaluate on a Custom Folder

Set `CUSTOM_AUDIO_DIR` to any folder containing `.flac` or `.wav` files. Transcripts must be in `*.trans.txt` files in the same LibriSpeech format.

In [9]:
# ── Uncomment and edit this cell to run on a custom folder ─────────────────
# CUSTOM_AUDIO_DIR = Path("/path/to/your/custom/audio")
# custom_ds = parse_librispeech_split(CUSTOM_AUDIO_DIR.parent, CUSTOM_AUDIO_DIR.name)
# custom_preds, custom_refs = [], []
# for i in range(0, len(custom_ds), BATCH_SIZE):
#     batch = custom_ds[i : i + BATCH_SIZE]
#     custom_preds.extend(transcribe_batch([a["array"] for a in batch["audio"]]))
#     custom_refs.extend(batch["sentence"])
# custom_wer = wer_metric.compute(predictions=custom_preds, references=custom_refs)
# custom_cer = cer_metric.compute(predictions=custom_preds, references=custom_refs)
# print(f"Custom WER: {custom_wer:.4f} | Custom CER: {custom_cer:.4f}")